In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/commodity_master.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nData types:")
print(df.dtypes)

Loaded: 413 rows, 9 columns

Missing values:
date                0
coal_price_usd      0
nickel_price_usd    0
usd_idr             0
coal_price_idr      0
nickel_price_idr    0
year                0
month               0
month_name          0
dtype: int64

Data types:
date                datetime64[ns]
coal_price_usd             float64
nickel_price_usd           float64
usd_idr                    float64
coal_price_idr             float64
nickel_price_idr           float64
year                         int64
month                        int64
month_name                  object
dtype: object


In [4]:
# Sort by date first
df = df.sort_values('date').reset_index(drop=True)

# 1. Price changes - month over month
df['coal_mom_change'] = df['coal_price_usd'].pct_change() * 100
df['nickel_mom_change'] = df['nickel_price_usd'].pct_change() * 100

# 2. Year over year change
df['coal_yoy_change'] = df['coal_price_usd'].pct_change(12) * 100
df['nickel_yoy_change'] = df['nickel_price_usd'].pct_change(12) * 100

# 3. Rolling averages (3 month and 12 month)
df['coal_ma3'] = df['coal_price_usd'].rolling(3).mean()
df['coal_ma12'] = df['coal_price_usd'].rolling(12).mean()
df['nickel_ma3'] = df['nickel_price_usd'].rolling(3).mean()
df['nickel_ma12'] = df['nickel_price_usd'].rolling(12).mean()

# 4. Price volatility (12 month rolling std)
df['coal_volatility'] = df['coal_price_usd'].rolling(12).std()
df['nickel_volatility'] = df['nickel_price_usd'].rolling(12).std()

# 5. Commodity price ratio (coal/nickel relationship)
df['coal_nickel_ratio'] = df['coal_price_usd'] / df['nickel_price_usd']

# 6. Price regime - is price above or below 12 month average?
df['coal_above_ma12'] = (df['coal_price_usd'] > df['coal_ma12']).astype(int)
df['nickel_above_ma12'] = (df['nickel_price_usd'] > df['nickel_ma12']).astype(int)

print("New features created:")
print(df.columns.tolist())

New features created:
['date', 'coal_price_usd', 'nickel_price_usd', 'usd_idr', 'coal_price_idr', 'nickel_price_idr', 'year', 'month', 'month_name', 'coal_mom_change', 'nickel_mom_change', 'coal_yoy_change', 'nickel_yoy_change', 'coal_ma3', 'coal_ma12', 'nickel_ma3', 'nickel_ma12', 'coal_volatility', 'nickel_volatility', 'coal_nickel_ratio', 'coal_above_ma12', 'nickel_above_ma12']


In [5]:
# Label major global events that affected commodity prices
# This adds business context to the data
def label_market_event(date):
    if date < pd.Timestamp('2000-01-01'):
        return 'Pre-2000'
    elif date < pd.Timestamp('2003-01-01'):
        return 'Post-dot-com crash'
    elif date < pd.Timestamp('2008-06-01'):
        return 'China boom'
    elif date < pd.Timestamp('2009-06-01'):
        return 'GFC crisis'
    elif date < pd.Timestamp('2016-01-01'):
        return 'Post-GFC recovery'
    elif date < pd.Timestamp('2020-03-01'):
        return 'Stable period'
    elif date < pd.Timestamp('2021-06-01'):
        return 'COVID crash'
    elif date < pd.Timestamp('2023-01-01'):
        return 'Post-COVID surge'
    else:
        return 'Current period'

df['market_period'] = df['date'].apply(label_market_event)

print("Market periods:")
print(df['market_period'].value_counts())

Market periods:
market_period
Pre-2000              96
Post-GFC recovery     79
China boom            65
Stable period         50
Current period        41
Post-dot-com crash    36
Post-COVID surge      19
COVID crash           15
GFC crisis            12
Name: count, dtype: int64


In [6]:
# Drop rows with NaN from rolling calculations
df_clean = df.dropna().reset_index(drop=True)

print(f"Before cleaning: {len(df)} rows")
print(f"After cleaning: {len(df_clean)} rows")
print(f"Removed: {len(df) - len(df_clean)} rows (rolling window warmup)")

df_clean.to_csv('../data/commodity_clean.csv', index=False)
print(f"\nSaved! Final shape: {df_clean.shape}")
print(f"Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")

Before cleaning: 413 rows
After cleaning: 401 rows
Removed: 12 rows (rolling window warmup)

Saved! Final shape: (401, 23)
Date range: 1993-01-01 00:00:00 to 2026-05-01 00:00:00
